# Aula 15 · Ajuste não linear

Esta aula apresenta o [capítulo 15 do site](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/). A ideia central: **quando a reta não basta, o ajuste continua sendo mínimos quadrados** — com um polinômio, com uma troca de variável que endireita a curva, ou com um ajuste direto. E um modelo flexível demais passa a ajustar o ruído.

**Ao fim da aula você consegue:**

1. ajustar polinômios e escolher o grau pelo erro nos dados de teste;
2. linearizar exponenciais e potências com logaritmos;
3. ajustar um modelo qualquer com `curve_fit`;
4. conferir o ajuste pelos resíduos.

**Roteiro:** 🧩 · 1. o polinômio · 2. sobreajuste · 3. 🧑‍🏫 linearização · 4. o ajuste direto · 5. outra área · 🎯 prática · 🧩 o Wi-Fi · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

# --- dados desta aula (baixados do site, se ainda não estiverem aqui) ---
import os
import urllib.request

for ARQUIVO in ["co2_mauna_loa.csv", "latencia_servidor.csv", "casos_acumulados.csv", "rssi_distancia.csv"]:
    if not os.path.exists(ARQUIVO):
        urllib.request.urlretrieve("https://lacouth.github.io/metodos_telecom-site/dados/" + ARQUIVO, ARQUIVO)
    print(ARQUIVO, "pronto")

## 🧩 O problema da aula

> **Telecomunicações — até onde vai o Wi-Fi?**
>
> *A equipe de TI de um campus mediu, com um celular, a potência do sinal de um ponto
> de acesso em várias distâncias. O celular perde a conexão abaixo de **−90 dBm**. O
> coordenador pergunta: "**qual o raio de cobertura de cada ponto de acesso, para
> saber quantos precisamos comprar?**"*

O sinal cai com o **logaritmo** da distância, e as paredes espalham as medições. No
fim da aula, você ajusta o modelo de propagação e calcula o raio.

## 1. Quando a reta não basta: o polinômio

No capítulo 14, a reta do CO₂ deixou resíduos em U. Um polinômio de grau 2 ainda é
**linear nos coeficientes**: `np.polyfit` com grau 2.

📖 [capítulo 15 · Quando a reta não basta: o polinômio](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#quando-a-reta-nao-basta-o-polinomio)

In [ ]:
# 📦 dados prontos — só rode esta célula
dados_co2 = np.loadtxt("co2_mauna_loa.csv", delimiter=",", skiprows=1)
ano = dados_co2[:, 0]
co2 = dados_co2[:, 1]
t = ano - 1960          # anos desde 1960 (números pequenos: a conta fica precisa)

### 🎯 Sua vez — Avaliar o polinômio do polyfit

Escreva `avalia_polinomio(coef, x)`, que calcula o polinômio com os coeficientes na ordem do `polyfit` (do grau mais alto para o mais baixo).

In [ ]:
def avalia_polinomio(coef, x):
    # sua solução aqui
    pass

In [ ]:
confere(avalia_polinomio, [
    (([2.0, 3.0, 1.0], 2.0), 15.0),
    (([1.0, 0.0], 5.0), 5.0),
])

<details>
<summary><b>💡 Dica</b></summary>

O grau é `len(coef) - 1`; o coeficiente `coef[k]` multiplica `x**(grau - k)`.

</details>

**✍️ Passo 1.** Ajuste `coef = np.polyfit(t, co2, 2)`, calcule os resíduos com a sua `avalia_polinomio` e desenhe-os contra `ano`, com a linha do zero.

In [ ]:
# ✍️ passo 1

**Preveja:** o U dos resíduos da reta vai continuar aparecendo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: os resíduos oscilam sem padrão, entre −1 e 1,4 ppm. O termo $0{,}013\,t^2$
é a aceleração que a reta não via.

📖 [capítulo 15 · Quando a reta não basta: o polinômio](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#quando-a-reta-nao-basta-o-polinomio)

</details>

**✍️ Passo 2.** Ache o ano em que a parábola passa de 450 ppm: a raiz de `avalia_polinomio(coef, tt) - 450`, com o `brentq` (capítulo 4) entre `tt = 60` e `tt = 120`. Some 1960.

In [ ]:
# ✍️ passo 2

**Preveja:** vai dar antes ou depois do 2047 que a reta previu na Aula 14?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**2034** — treze anos antes. A reta, que não acelera, atrasava a previsão. A
resposta para a jornalista da Aula 14 é essa (supondo que o ritmo de
aceleração continue).

</details>

## 2. Sobreajuste

A latência do servidor, separada em treino (posições pares) e teste (ímpares).

📖 [capítulo 15 · Sobreajuste](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#sobreajuste)

In [ ]:
# 📦 dados prontos — só rode esta célula
dados_lat = np.loadtxt("latencia_servidor.csv", delimiter=",", skiprows=1)
x = dados_lat[:, 0] / 100      # centenas de usuários
y = dados_lat[:, 1]
x_treino = x[0::2]
y_treino = y[0::2]
x_teste = x[1::2]
y_teste = y[1::2]

**✍️ Passo 3.** Para cada grau em `[1, 2, 4, 6, 8, 10]`, ajuste no treino e imprima o erro (a raiz da média dos quadrados, `np.sqrt(np.mean((previsto - y)**2))`) no treino e no teste.

In [ ]:
# ✍️ passo 3

**Preveja:** com grau maior, o erro no teste também cai?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Não**: no treino cai de 2,7 para 0,7 ms; no teste **sobe** de 2,4 para 4,0 ms. O
polinômio de grau alto decorou o ruído do treino. É o fenômeno de Runge vestido
de estatística. Escolha o grau pelo erro no **teste**.

📖 [capítulo 15 · Sobreajuste](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#sobreajuste)

</details>

> ⚠️ **Armadilha.** Avaliar um modelo nos mesmos dados que o ajustaram sempre dá uma impressão boa
demais. Separe um pedaço dos dados que o modelo nunca viu.

## 3. No quadro: a linearização

📖 [capítulo 15 · No quadro: a linearização](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#no-quadro-a-linearizacao)

### 🧑‍🏫 No quadro — linearizando exponenciais e potências

Caderno de papel aberto. No quadro:

1. $y = a\,e^{bx}$: logaritmo dos dois lados → reta entre $x$ e $\ln y$;
2. $y = a\,x^b$: logaritmo dos dois lados → reta entre $\ln x$ e $\ln y$;
3. ajustar a reta e desfazer: $a = e^{\text{intercepto}}$;
4. a meia-vida: $\ln 2 / k$.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ \ln y = \ln a + b\,x \qquad \ln y = \ln a + b\,\ln x $$

</details>

**✍️ Passo 4.** Ajuste a exponencial das contagens do iodo-131: `dia = np.array([0, 2, 4, 6, 8, 10, 12, 14, 16])` e as contagens `[10020, 8440, 7010, 5950, 4970, 4230, 3500, 2980, 2510]` — com `np.polyfit(dia, np.log(contagem), 1)`. Calcule a meia-vida.

In [ ]:
# ✍️ passo 4

**Preveja:** a meia-vida do iodo-131 é 8,02 dias. Quanto vai dar?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

8,00 dias, com $k = 0{,}0866$ por dia. Com a troca de variável, a exponencial
virou reta, e o `polyfit` de grau 1 bastou.

📖 [capítulo 15 · No quadro: a linearização](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#no-quadro-a-linearizacao)

</details>

> 🧰 **Comando novo: `np.log10`**
>
> `np.log10(x)` é o logaritmo na base 10, o dos decibéis: `np.log10(1000)` é 3. Dá o
> mesmo que `np.log(x) / np.log(10)`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(np.log10(1000), np.log10(2))

## 4. Um modelo que não se lineariza

A logística de uma epidemia, $K/(1 + A\,e^{-rt})$, não vira reta com logaritmo
nenhum. O ajuste direto procura os parâmetros que minimizam a soma dos quadrados.

📖 [capítulo 15 · Um modelo que não se lineariza](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#um-modelo-que-nao-se-lineariza)

> 🧰 **Comando novo: `from scipy.optimize import curve_fit`**
>
> `curve_fit(modelo, x, y, p0=[...])` ajusta os parâmetros de `modelo(x, p1, p2, ...)`.
> O primeiro argumento do modelo é o $x$; os outros são os parâmetros. `p0=` são os
> **chutes iniciais**, um por parâmetro. Ele devolve os parâmetros e uma matriz de
> incerteza: `parametros, cov = curve_fit(...)`.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
from scipy.optimize import curve_fit


def reta(x, a, b):
    return a + b * x


parametros, cov = curve_fit(reta, [0, 1, 2, 3], [1.0, 3.1, 4.9, 7.0], p0=[0, 1])
print(parametros)

**✍️ Passo 5.** Leia `casos_acumulados.csv`, escreva `logistica(t, K, A, r)` e ajuste com `curve_fit(logistica, dia, casos, p0=[10000, 100, 0.1])`. Imprima os parâmetros e o dia do pico, $\ln A / r$.

In [ ]:
# ✍️ passo 5

**Preveja:** o pico ajustado fica perto do dia 50 do modelo que gerou os dados?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Dia 49,8 (o modelo diz 49,9), com um total de ~11 900 casos. No capítulo 3, a
derivada do dado com ruído dizia dia 47: ajustar um modelo **filtra** o ruído
que a derivada amplificava.

📖 [capítulo 15 · Um modelo que não se lineariza](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#um-modelo-que-nao-se-lineariza)

</details>

## 5. Mesmo método, outra área

**Biologia.** Massa (kg) e metabolismo basal (W) de sete animais, do camundongo ao
elefante: `[0.021, 0.29, 3.0, 15.0, 70.0, 500.0, 4000.0]` e
`[0.21, 1.45, 8.3, 30.0, 80.0, 350.0, 2300.0]`.

📖 [capítulo 15 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#mesmo-metodo-outra-area)

In [ ]:
# 📦 dados prontos — só rode esta célula
massa = np.array([0.021, 0.29, 3.0, 15.0, 70.0, 500.0, 4000.0])
metabolismo = np.array([0.21, 1.45, 8.3, 30.0, 80.0, 350.0, 2300.0])

**✍️ Passo 6.** Ajuste a potência $M = a\,m^b$ pela reta de `np.log(metabolismo)` contra `np.log(massa)`.

In [ ]:
# ✍️ passo 6

**Preveja:** o expoente fica perto de 1 (proporcional à massa) ou de outro valor?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$b = 0{,}755$: a **lei de Kleiber**, expoente 3/4. Um animal 10 vezes mais pesado
gasta 5,7 vezes mais energia, e não 10.

📖 [capítulo 15 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade7-regressao/15-ajuste-nao-linear/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

A prática desta aula é o problema, logo abaixo: uma linearização com logaritmo na base 10, em decibéis.

## 🧩 Resolvendo o problema

> *"**Qual o raio de cobertura de cada ponto de acesso?**"* — o coordenador de TI.

O modelo de propagação **log-distância** diz que a potência recebida cai assim:

$$ \text{RSSI}(d) = A - 10\,n\,\log_{10}(d), $$

em que $A$ é a potência a 1 m e $n$ é o **expoente de perda de percurso**: 2 no espaço
livre, de 2,5 a 4 dentro de prédios. É uma **reta** entre $\log_{10} d$ e o RSSI.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Potência recebida (dBm) de um ponto de acesso Wi-Fi, medida com o celular em
# várias distâncias (m) num campus.
dados_rssi = np.loadtxt("rssi_distancia.csv", delimiter=",", skiprows=1)
distancia = dados_rssi[:, 0]
rssi = dados_rssi[:, 1]
SENSIBILIDADE = -90.0     # abaixo disso, o celular perde a conexão (dBm)

plt.figure()
plt.plot(distancia, rssi, "o")
plt.xlabel("distância (m)")
plt.ylabel("potência recebida (dBm)")
plt.grid()
plt.show()

### 🎯 Sua vez — O modelo de propagação

Escreva `modelo_propagacao(distancia, rssi)`, que ajusta a reta de `rssi` contra
`np.log10(distancia)` com `np.polyfit` e devolve a tupla `(A, n)`: `A` é o
intercepto e `n` é **menos a inclinação dividida por 10**.

In [ ]:
def modelo_propagacao(distancia, rssi):
    # sua solução aqui
    pass

In [ ]:
confere(modelo_propagacao, [
    ((np.array([1.0, 10.0, 100.0]), np.array([-40.0, -60.0, -80.0])), (-40.0, 2.0)),
    ((distancia, rssi), (-42.431833124248485, 2.5274116293854023)),
], tol=1e-9)

<details>
<summary><b>💡 Dica</b></summary>

`coef = np.polyfit(np.log10(distancia), rssi, 1)`; a inclinação é `coef[0]` e vale $-10n$.

</details>

O raio de cobertura é a distância em que o RSSI chega à sensibilidade: $d = 10^{(A - \text{sensibilidade})/(10n)}$.

In [ ]:
resposta = modelo_propagacao(distancia, rssi)
if resposta is not None:
    A, n = resposta
    raio = 10 ** ((A - SENSIBILIDADE) / (10 * n))
    print("A =", A, "dBm a 1 m;  n =", n)
    print("raio de cobertura:", raio, "m")

<details>
<summary><b>▶ O que os números dizem</b></summary>

O expoente ajustado é **n = 2.53**: entre o espaço livre (2) e um prédio
cheio de paredes (4), como se espera num campus. O raio de cobertura sai **76 m**.

Na prática, as operadoras usam uma **margem**: por causa das variações (as paredes
espalham as medições em ±4 dB), projetam para uma sensibilidade uns 10 dB acima (−80
dBm), o que dá um raio bem menor. Refaça a conta com −80 dBm: o raio cai para menos da
metade, e o número de pontos de acesso para cobrir o campus, que cresce com o
**quadrado** do inverso do raio, mais que quadruplica. Um parâmetro ajustado de uma
reta decide uma compra.

</details>

## 📋 A lista

Abra a [Lista 15](https://lacouth.github.io/metodos_telecom-site/listas/lista15/). O **Exercício 01** é à mão (✏️): uma exponencial que vira
reta. Comece por ele, no papel.

**a)** Quanto vale $\ln 10 - \ln 5$? E $\ln 20 - \ln 10$?

<details>
<summary><b>▶ Resposta</b></summary>

Os dois valem $\ln 2 = 0{,}6931$: diferenças iguais, pontos numa reta.

</details>

Termine o exercício e siga para o **Exercício 02**, o ajuste exponencial como função.

## 🚪 Antes de sair

**1.** Por que um polinômio de grau 2 ainda é "ajuste linear"?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque é linear nos **coeficientes**: as equações normais continuam sendo um sistema linear, mesmo com $t^2$ no modelo.

</details>

**2.** Como saber se o grau escolhido está sobreajustando?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Comparando o erro no treino com o erro em dados de teste que o modelo não viu: se o do teste sobe enquanto o do treino cai, é sobreajuste.

</details>

**3.** Por que o modelo do Wi-Fi é uma reta em $\log_{10} d$?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Porque a potência cai com uma potência da distância ($d^{-n}$), e em decibéis (logaritmo) uma potência vira produto: $-10n\log_{10} d$.

</details>

## 🏠 Para casa

- Termine a [Lista 15](https://lacouth.github.io/metodos_telecom-site/listas/lista15/).
- Leia o começo do [capítulo 16](https://lacouth.github.io/metodos_telecom-site/unidade8-series-temporais/16-series-temporais/):
  e quando o $x$ é o próprio **tempo**?